In [ ]:
import kagglehub
%pip install catboost


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd
# Read the dataset Q3_data.csv using read_csv()

q3 = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(q3)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

# Handle missing values appropriately
missing = pd.DataFrame([df.isna().sum(), df.isna().sum() / len(df)],
                index=['missing count', 'missing percentage']).T

print(missing.loc['Target'])
# Target has no missing values

missing['missing percentage'].hist()
plt.xlabel('missing values percentage')
plt.ylabel('frequency')
plt.show()
# missing values has no clear structure, fill with zeros

df = df.fillna(0)

df.isna().sum().sum()

In [ ]:
# Task 2: Write your code here:

# Check and remove duplicates if any exist
if (df.duplicated().sum() > 0):
  df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:

# Encode categorical variables if needed
catag_feats = df.select_dtypes('object').columns
len(catag_feats)
print('no catagorical variables')

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
# Apply feature scaling to numerical features (Use StandardScaler)
ss = StandardScaler()

# don't scale the target
to_scale = df.drop('Target', axis=1).columns
df[to_scale] = ss.fit_transform(df[to_scale])

df

In [ ]:
# Task 5: Write your code here:

# Check for target imbalance and state if it is imbalanced or not
df['Target'].value_counts(normalize=True)

# data is imbalanced, we'll need to put that in consideration
# (use stratified KFold, f1_score metric, etc..)

In [ ]:
# Task 1: Write your code here:

# Split the dataset into features (X) and target (y)
X = df.drop('Target', axis=1)
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Use the correct split: KFold OR StratifiedKFold
from sklearn.model_selection import StratifiedKFold
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits)

#Train a CatBoostClassifier model
from catboost import CatBoostClassifier
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

accuracy = []
f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Evaluate using the appropriate metric only (Accuracy vs. F1 Score).
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate
  accuracy.append(accuracy_score(y_test, y_pred))
  f1.append(f1_score(y_test, y_pred, zero_division=0))

# Print the averaged score across all folds
print('average accuracy score', np.mean(accuracy))
print('average f1 score', np.mean(f1))

In [ ]:
# Task 1: Write your code here:

# Feature importance
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='blue')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Identify and print the name of the most important feature (the 'golden feature')

print(importance['feature'][importance['importance'].argmax()])

In [ ]:
X = X['P_2']

# Use the correct split: KFold OR StratifiedKFold
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits)

#Train a CatBoostClassifier model
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

accuracy_p_2 = []
f1_p_2 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Evaluate using the appropriate metric only (Accuracy vs. F1 Score).
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate
  accuracy_p_2.append(accuracy_score(y_test, y_pred))
  f1_p_2.append(f1_score(y_test, y_pred, zero_division=0))

# Print the averaged score across all folds
print('average accuracy score', np.mean(accuracy))
print('average f1 score', np.mean(f1))